In [1]:
!pip install lizard pydriller tqdm pandas requests


   ------------- -------------------------- 1/3 [lizard]
   -------------------------- ------------- 2/3 [pydriller]
   ---------------------------------------- 3/3 [pydriller]



In [2]:
import re
import requests
import pandas as pd
from collections import defaultdict, Counter
from itertools import islice
from pydriller import Repository
from tqdm import tqdm
import lizard

REPO_URL = "https://github.com/django/django"
OUTPUT_FILE = "django_osv_dataset_5000.csv"
FEATURES_FILE = "django_osv_features_5000.csv"

PACKAGE_NAME = "Django"
ECOSYSTEM = "PyPI"

SAFE_KEYWORDS = {
    "refactor", "cleanup", "clean up", "typo", "docs", "documentation",
    "style", "format", "test", "tests", "ci", "build", "bump", "version",
    "rename", "comment", "lint", "readme", "translation", "chore"
}


In [3]:
def load_osv_records(package_name=PACKAGE_NAME, ecosystem=ECOSYSTEM):
    payload = {
        "queries": [
            {
                "package": {
                    "name": package_name,
                    "ecosystem": ecosystem
                }
            }
        ]
    }

    response = requests.post(
        "https://api.osv.dev/v1/querybatch",
        json=payload,
        timeout=60
    )
    response.raise_for_status()
    batch_data = response.json()

    vuln_ids = []
    for item in batch_data.get("results", []):
        for vuln in item.get("vulns", []):
            if "id" in vuln:
                vuln_ids.append(vuln["id"])

    records = []
    for vuln_id in tqdm(vuln_ids, desc="Loading OSV advisories"):
        r = requests.get(f"https://api.osv.dev/v1/vulns/{vuln_id}", timeout=60)
        if r.status_code == 200:
            records.append(r.json())

    return records


def parse_osv_data(records):
    vuln_hashes = set()
    vuln_keywords = set()
    vuln_titles = {}

    for record in records:
        vuln_id = record.get("id", "")
        text_parts = []

        summary = record.get("summary", "") or ""
        details = record.get("details", "") or ""
        text_parts.extend([summary, details])

        if summary:
            vuln_titles[vuln_id] = summary

        for ref in record.get("references", []):
            url = ref.get("url", "")
            text_parts.append(url)

            commit_match = re.search(r"/commit/([0-9a-f]{7,40})", url)
            if commit_match:
                vuln_hashes.add(commit_match.group(1))

        for affected in record.get("affected", []):
            for rng in affected.get("ranges", []):
                if rng.get("type") == "GIT":
                    for event in rng.get("events", []):
                        fixed_hash = event.get("fixed")
                        if fixed_hash:
                            vuln_hashes.add(fixed_hash)

        full_text = " ".join(text_parts).lower()
        for word in re.findall(r"[a-zA-Z][a-zA-Z\-]{3,}", full_text):
            if word in {
                "security", "vulnerability", "injection", "xss", "csrf",
                "sqli", "bypass", "exposure", "disclosure", "traversal",
                "rce", "overflow", "deserialization", "spoofing",
                "privilege", "escalation", "authentication", "authorization",
                "redirect", "dos", "denial", "forgery", "cve"
            }:
                vuln_keywords.add(word)

    return vuln_hashes, vuln_keywords, vuln_titles


osv_records = load_osv_records()
VULN_HASHES, VULN_KEYWORDS, OSV_TITLES = parse_osv_data(osv_records)

print("OSV advisories:", len(osv_records))
print("Known fix hashes:", len(VULN_HASHES))
print("Vulnerability keywords:", sorted(VULN_KEYWORDS))


Loading OSV advisories: 100%|████████████████████████████████████████████████████████| 267/267 [01:12<00:00,  3.68it/s]

OSV advisories: 267
Known fix hashes: 339
Vulnerability keywords: ['authentication', 'bypass', 'csrf', 'denial', 'disclosure', 'exposure', 'forgery', 'injection', 'redirect', 'security', 'spoofing', 'traversal', 'vulnerability']


In [4]:
def classify_commit(commit_hash, message):
    short_hash = (commit_hash or "")[:40]

    if short_hash in VULN_HASHES:
        return "vuln"

    if not message or len(message.strip()) < 4:
        return "skip"

    msg = message.lower()

    has_vuln_keyword = any(word in msg for word in VULN_KEYWORDS) or ("security" in msg) or ("cve-" in msg)
    has_safe_keyword = any(word in msg for word in SAFE_KEYWORDS)

    if has_vuln_keyword:
        return "vuln"

    if has_safe_keyword:
        return "safe"

    return "safe"


In [5]:
class DjangoDatasetBuilder:
    def __init__(self, repo_url):
        self.repo_url = repo_url
        self.rows = []
        self.file_history = defaultdict(lambda: {
            "commits": 0,
            "authors": set(),
            "security_commits": 0,
            "first_seen": None
        })
        self.counters = Counter()

    def calc_code_metrics(self, code, filename):
        if not code or not str(code).strip():
            return None

        try:
            analysis = lizard.analyze_file.analyze_source_code(filename, code)
        except Exception:
            return None

        if not analysis or analysis.nloc == 0:
            return None

        funcs = analysis.function_list
        avg_ccn = sum(f.cyclomatic_complexity for f in funcs) / len(funcs) if funcs else 0
        max_ccn = max((f.cyclomatic_complexity for f in funcs), default=0)
        avg_params = sum(f.parameter_count for f in funcs) / len(funcs) if funcs else 0

        return {
            "loc": analysis.nloc,
            "func_cnt": len(funcs),
            "ccn": analysis.CCN,
            "avg_ccn": round(avg_ccn, 3),
            "max_ccn": max_ccn,
            "token_count": len(re.findall(r"\w+", code)),
            "avg_params": round(avg_params, 3)
        }

    def build_row(self, commit, modified_file, label):
        path = modified_file.new_path or modified_file.old_path
        if not path:
            return None

        ext = "." + path.split(".")[-1].lower() if "." in path else ""
        if ext not in {".py", ".js", ".ts", ".java", ".scala"}:
            return None

        code = modified_file.source_code_before if label == "vuln" else (modified_file.source_code or modified_file.source_code_before)
        metrics = self.calc_code_metrics(code, path)

        if not metrics:
            return None

        hist = self.file_history[path]

        file_age_days = 0
        if hist["first_seen"] is not None:
            file_age_days = max((commit.committer_date - hist["first_seen"]).days, 0)

        row = {
            "file": path,
            "hash": commit.hash,
            "date": commit.committer_date.date().isoformat(),
            "author": commit.author.name,
            "message": commit.msg[:150],
            "is_vuln": 1 if label == "vuln" else 0,
            "file_commits": hist["commits"],
            "unique_authors": len(hist["authors"]),
            "file_age_days": file_age_days,
            "security_touch_ratio": round(
                hist["security_commits"] / hist["commits"], 4
            ) if hist["commits"] else 0.0
        }
        row.update(metrics)
        return row

    def update_history(self, commit, modified_file, label):
        path = modified_file.new_path or modified_file.old_path
        if not path:
            return

        hist = self.file_history[path]
        hist["commits"] += 1
        hist["authors"].add(commit.author.email or commit.author.name)

        if hist["first_seen"] is None:
            hist["first_seen"] = commit.committer_date

        if label == "vuln":
            hist["security_commits"] += 1

    def extract(self, max_commits=70000):
        repo = Repository(self.repo_url, only_no_merge=True)
        for commit in tqdm(islice(repo.traverse_commits(), max_commits), total=max_commits, desc="Mining Django"):
            label = classify_commit(commit.hash, commit.msg)
            if label == "skip":
                self.counters["skipped_commits"] += 1
                continue

            for modified_file in commit.modified_files:
                row = self.build_row(commit, modified_file, label)
                if row is not None:
                    self.rows.append(row)
                    self.counters["rows"] += 1
                    self.counters[f"{label}_rows"] += 1

                self.update_history(commit, modified_file, label)

        return pd.DataFrame(self.rows)


In [6]:
def make_dataset_5000(df, target_total=5000, vuln_share=0.10):
    vuln = df[df["is_vuln"] == 1].copy()
    safe = df[df["is_vuln"] == 0].copy()

    need_vuln = int(target_total * vuln_share)
    need_safe = target_total - need_vuln

    print("Raw rows:")
    print("  vuln =", len(vuln))
    print("  safe =", len(safe))
    print("Target rows:")
    print("  vuln =", need_vuln)
    print("  safe =", need_safe)

    if len(vuln) == 0 or len(safe) == 0:
        raise ValueError("Not enough rows to build the final dataset")

    vuln_part = vuln.sample(n=need_vuln, replace=len(vuln) < need_vuln, random_state=42)
    safe_part = safe.sample(n=need_safe, replace=len(safe) < need_safe, random_state=42)

    out = pd.concat([vuln_part, safe_part], ignore_index=True)
    out = out.sample(frac=1, random_state=42).reset_index(drop=True)
    return out


In [7]:
builder = DjangoDatasetBuilder(REPO_URL)
df_raw = builder.extract(max_commits=70000)

print("Collected rows:", len(df_raw))
df_raw.head()


Mining Django:  48%|█████████████████████████████▍                               | 33846/70000 [47:00<50:12, 12.00it/s]


Collected rows: 78171


,file,hash,date,author,message,is_vuln,file_commits,unique_authors,file_age_days,security_touch_ratio,loc,func_cnt,ccn,avg_ccn,max_ccn,token_count,avg_params
0,django-docs\build.py,07ffc7d605cc96557db28a9e35da69bc0719611b,2005-07-13,Adrian Holovaty,Imported fledgeling Django documentation from ...,0,0,0,0,0.0,36,2,4,2.000,3,141,1.500
1,django\bin\daily_cleanup.py,ed114e15106192b22ebb78ef5bf5bce72b419d13,2005-07-13,Adrian Holovaty,Imported Django from private SVN repository (c...,0,0,0,0,0.0,10,1,1,1.000,1,51,0.000
2,django\bin\django-admin.py,ed114e15106192b22ebb78ef5bf5bce72b419d13,2005-07-13,Adrian Holovaty,Imported Django from private SVN repository (c...,0,0,0,0,0.0,355,21,82,3.905,18,2184,1.286
3,django\bin\profiling\gather_profile_stats.py,ed114e15106192b22ebb78ef5bf5bce72b419d13,2005-07-13,Adrian Holovaty,Imported Django from private SVN repository (c...,0,0,0,0,0.0,25,1,6,6.000,6,127,1.000
4,django\bin\profiling\handler.py,ed114e15106192b22ebb78ef5bf5bce72b419d13,2005-07-13,Adrian Holovaty,Imported Django from private SVN repository (c...,0,0,0,0,0.0,8,1,1,1.000,1,125,1.000


In [8]:
df = make_dataset_5000(df_raw, target_total=5000, vuln_share=0.10)

df.to_csv(OUTPUT_FILE, index=False)

feature_cols = [
    "loc", "func_cnt", "ccn", "avg_ccn", "max_ccn", "token_count",
    "avg_params", "file_commits", "unique_authors",
    "file_age_days", "security_touch_ratio", "is_vuln"
]
df[feature_cols].to_csv(FEATURES_FILE, index=False)

print("=" * 45)
print("Final dataset:", OUTPUT_FILE)
print("=" * 45)
print("Total rows      :", len(df))
print("Vulnerable rows :", int(df['is_vuln'].sum()))
print("Safe rows       :", int((df['is_vuln'] == 0).sum()))
print("Unique files    :", df['file'].nunique())
print("Period          :", df['date'].min(), "->", df['date'].max())
print("Columns         :", len(df.columns))


Raw rows:
  vuln = 1529
  safe = 76642
Target rows:
  vuln = 500
  safe = 4500
Final dataset: django_osv_dataset_5000.csv
Total rows      : 5000
Vulnerable rows : 500
Safe rows       : 4500
Unique files    : 1814
Period          : 2005-07-15 -> 2026-03-24
Columns         : 17


In [9]:
group_cols = [
    "loc", "func_cnt", "ccn", "avg_ccn", "max_ccn",
    "token_count", "avg_params",
    "file_commits", "unique_authors", "file_age_days",
    "security_touch_ratio"
]

comparison = df.groupby("is_vuln")[group_cols].mean().round(3).T
comparison.columns = ["Safe (0)", "Vulnerable (1)"]
comparison["Diff %"] = (
    (comparison["Vulnerable (1)"] - comparison["Safe (0)"])
    / comparison["Safe (0)"].replace(0, 1) * 100
).round(1).astype(str) + "%"

comparison


,Safe (0),Vulnerable (1),Diff %
loc,389.257,412.866,6.1%
func_cnt,34.664,39.980,15.3%
ccn,75.418,68.572,-9.1%
avg_ccn,2.400,2.112,-12.0%
max_ccn,8.681,6.898,-20.5%
token_count,2342.509,2426.102,3.6%
avg_params,1.586,1.611,1.6%
file_commits,52.663,53.940,2.4%
unique_authors,16.892,20.290,20.1%
file_age_days,1720.044,2141.922,24.5%


In [10]:
print("Examples of vulnerable rows:")
display(df[df["is_vuln"] == 1].head(3))

print("\nExamples of safe rows:")
display(df[df["is_vuln"] == 0].head(3))

try:
    from google.colab import files
    files.download(OUTPUT_FILE)
    files.download(FEATURES_FILE)
except Exception:
    print("If you are not in Colab, files are already saved locally.")


Examples of vulnerable rows:


,file,hash,date,author,message,is_vuln,file_commits,unique_authors,file_age_days,security_touch_ratio,loc,func_cnt,ccn,avg_ccn,max_ccn,token_count,avg_params
5,django\middleware\csrf.py,720888a14699a80a6cd07d32514b9dcd5b1005fb,2013-02-07,Aymeric Augustin,Fixed #15808 -- Added optional HttpOnly flag t...,1,29,10,1199,0.6897,114,8,25,3.125,13,900,1.875
11,tests\check_framework\test_security.py,64331419c848a8b6835b25f2c600d7150f722c96,2021-01-12,Hasan Ramezani,Fixed #32311 -- Added system check for CSRF_FA...,1,16,12,2313,0.1250,297,53,54,1.019,2,1292,1.000
27,tests\regressiontests\csrf_tests\tests.py,d51fb74360b94f2a856573174f8aae3cd905dd35,2013-02-19,Carl Meyer,Added a new required ALLOWED_HOSTS setting for...,1,30,9,1211,0.5667,193,34,35,1.029,2,1254,1.029



Examples of safe rows:


,file,hash,date,author,message,is_vuln,file_commits,unique_authors,file_age_days,security_touch_ratio,loc,func_cnt,ccn,avg_ccn,max_ccn,token_count,avg_params
0,django\conf\urls\static.py,610876661ecdfbc1a3ca3c5458e1801313501c77,2017-02-22,Anton Samarchyan,Fixed typos in code comments.,0,9,6,2200,0.0,13,1,4,4.000,4,113,3.000
1,tests\schema\tests.py,34b411762b50883d768d7b67e0a158ec39da8b09,2023-10-30,Tom Carrick,Fixed #34932 -- Restored varchar_pattern_ops/t...,0,250,75,3845,0.0,4290,241,432,1.793,13,19233,1.187
2,django\contrib\formtools\tests\wizard\wizardte...,c4c2c99669ef9d58306856c47058b121bbecfe5f,2014-05-30,Florian Apolloner,Fixed #19905 -- Fixed leakage of file descript...,0,16,10,1021,0.0,349,26,28,1.077,2,1753,1.231


If you are not in Colab, files are already saved locally.
